# CS221 Homework 3: Route Planning

This notebook works through all five problems of the route planning homework.

- **Problem 1** (Written): Grid City -- cost analysis and UCS behavior
- **Problem 2** (Code + Written): Finding Shortest Paths with UCS
- **Problem 3** (Code + Written): Finding Shortest Paths with Unordered Waypoints
- **Problem 4** (Code): Speeding up Search with A*
- **Problem 5** (Written): AI Tutor sessions

Key concepts from Lecture 5 (Search) and Lecture 6 (A* Search, Relaxation) are applied throughout.

---
## Problem 1: Grid City (Written)

Consider a city modeled as an infinite 2D grid. You start at location $(0, 0)$ and want to reach location $(m, n)$ where $m, n \geq 0$.

- **Actions:** $\{(+1,0), (-1,0), (0,+1), (0,-1)\}$
- **Cost function:** $\text{Cost}((x,y), a) = 1 + \max(x, 0)$

The cost of moving increases with the $x$-coordinate -- moving from a location with a higher $x$ value is more expensive.

### 1a. Minimum Cost Path

**Task:** Find the minimum cost of reaching $(m, n)$ from $(0, 0)$. Describe one possible minimum cost path and whether it is unique.

**Answer:**

The cost to move from $(x, y)$ is $1 + \max(x, 0)$, regardless of direction. To minimize total cost, we want to spend as few steps as possible at high $x$ values. The optimal strategy is:

1. First move vertically from $(0,0)$ to $(0,n)$: each step costs $1 + 0 = 1$, total = $n$
2. Then move horizontally from $(0,n)$ to $(m,n)$: step $i$ moves from $(i-1, n)$ to $(i, n)$ at cost $1 + (i-1)$

$$\text{Total cost} = n + \sum_{i=0}^{m-1}(1 + i) = n + m + \frac{m(m-1)}{2}$$

**Uniqueness:** The path is **not** unique. Any path that keeps $x = 0$ for all vertical movement and then moves right achieves the same cost. For example, moving up partway, then right, then up more would be worse -- but interleaving vertical moves *before* any rightward moves doesn't change cost. Specifically, you could move to $(0, k)$, then to $(m, k)$, then to $(m, n)$, but the rightward segment would cost the same and the vertical segments at $x=0$ vs $x=m$ would differ. So the key constraint is: **do all vertical movement at $x=0$** (or at the lowest $x$ possible).

### 1b. UCS Behavior

**Task:** How will Uniform Cost Search behave on this problem? Mark true/false:

1. **UCS will never terminate** because the state space is infinite.
2. **UCS returns the minimum cost path**, exploring only $(x,y)$ where $0 \leq x \leq m$ and $0 \leq y \leq n$.
3. **UCS returns the minimum cost path**, exploring only locations with past costs $\leq$ minimum cost to $(m,n)$.

**Answer:**

1. **False.** UCS terminates even on infinite state spaces, as long as all action costs are positive (which they are here, since $\text{Cost} \geq 1$). UCS explores states in order of increasing past cost, so it will eventually reach $(m,n)$ and terminate.

2. **False.** UCS does return the minimum cost path, but it may explore states outside this rectangle. Since moving left is allowed and costs only $1 + \max(x, 0)$, UCS could explore states with negative $x$ values (where moving is cheap at cost $1$).

3. **True.** UCS explores states in order of increasing past cost. Any state with past cost exceeding the minimum cost to $(m,n)$ would be explored *after* $(m,n)$ is already dequeued, at which point UCS has terminated.

### 1c. General Graph Properties

**Task:** For arbitrary graphs, mark true/false:

1. Adding a connection between two nodes **cannot increase** the minimum cost between any two nodes.
2. Making an action's cost sufficiently small (possibly negative) **ensures** that action appears in the UCS solution path.
3. Increasing each action cost by 1 **doesn't change** the minimum cost path.

**Answer:**

1. **True.** Adding an edge only creates new possible paths; it never removes existing ones. The minimum cost is $\min$ over all paths, so adding options can only decrease or maintain it.

2. **False.** UCS requires non-negative edge weights to function correctly. With negative costs, UCS may not find the optimal path at all. Moreover, even conceptually, making one edge very cheap doesn't guarantee it lies on the optimal path -- the rest of the path through that edge could still be more expensive overall.

3. **False.** If each action costs 1 more, then a path with $k$ steps now costs $k$ more than before. Paths of different lengths are penalized differently. A longer path that was previously optimal might now be more expensive than a shorter alternative. For example, if path A has 2 steps at cost 3 each (total 6) and path B has 5 steps at cost 1.5 each (total 7.5), then after adding 1: path A costs 8 and path B costs 12.5 -- same winner. But if path A had cost 10 with 10 steps and path B had cost 10.5 with 2 steps, after adding 1: A costs 20, B costs 12.5 -- winner changes.

---
## Problem 2: Finding Shortest Paths

We now move from theory to code. The goal is to implement a search problem that finds the shortest path between locations on a map using Uniform Cost Search.

**From Lecture 5:** A search problem is defined by:
- `start_state()`: the initial state
- `successors(state)`: returns a list of `(action, cost, new_state)` triples
- `is_end(state)`: returns whether we've reached a goal state

UCS explores states in order of increasing past cost, guaranteeing optimality when all costs are non-negative.

### 2a. ShortestPathProblem Implementation

**Task:** Implement `ShortestPathProblem` with:
- **Input:** `city_map`, `start_location`, `end_tag`
- **Output:** minimum cost path from `start_location` to ANY location tagged with `end_tag`
- **Implement:** `start_state()`, `successors(state)`, `is_end(state)`

The action space consists of all named locations (i.e., the action is the name of the neighboring location you move to).

In [11]:
from util import State, Step, SearchProblem, UniformCostSearch
from map_util import CityMap

class ShortestPathProblem(SearchProblem):
    def __init__(self, city_map, start_location, end_tag):
        self.city_map = city_map
        self.start_location = start_location
        self.end_tag = end_tag

    def start_state(self):
        return State(location=self.start_location)

    def successors(self, state):
        result = []
        for neighbor, distance in self.city_map.distances[state.location].items():
            result.append(
                Step(
                    action=neighbor,
                    cost=distance,
                    state=State(location=neighbor)
                )
            )
        return result

    def is_end(self, state):
        return self.end_tag in self.city_map.tags[state.location]

### Test on a Synthetic Grid Map

Build a 5x5 grid with tagged locations and explore the map structure before running UCS.

In [12]:
from map_util import create_grid_map_with_custom_tags

tags = {(x, y): [] for x in range(5) for y in range(5)}
tags[(0, 0)] = ["landmark=start"]
tags[(4, 1)] = ["amenity=food"]
tags[(2, 4)] = ["amenity=food"]
tags[(3, 3)] = ["parking=garage"]

city_map = create_grid_map_with_custom_tags(5, 5, tags)
city_map

In [13]:
print(city_map.geo_locations.keys())
print(city_map.tags)
print(city_map.distances["0,0"])

dict_keys(['0,0', '0,1', '0,2', '0,3', '0,4', '1,0', '1,1', '1,2', '1,3', '1,4', '2,0', '2,1', '2,2', '2,3', '2,4', '3,0', '3,1', '3,2', '3,3', '3,4', '4,0', '4,1', '4,2', '4,3', '4,4'])
defaultdict(<class 'list'>, {'0,0': ['label=0,0', 'x=0', 'y=0', 'landmark=start'], '0,1': ['label=0,1', 'x=0', 'y=1'], '0,2': ['label=0,2', 'x=0', 'y=2'], '0,3': ['label=0,3', 'x=0', 'y=3'], '0,4': ['label=0,4', 'x=0', 'y=4'], '1,0': ['label=1,0', 'x=1', 'y=0'], '1,1': ['label=1,1', 'x=1', 'y=1'], '1,2': ['label=1,2', 'x=1', 'y=2'], '1,3': ['label=1,3', 'x=1', 'y=3'], '1,4': ['label=1,4', 'x=1', 'y=4'], '2,0': ['label=2,0', 'x=2', 'y=0'], '2,1': ['label=2,1', 'x=2', 'y=1'], '2,2': ['label=2,2', 'x=2', 'y=2'], '2,3': ['label=2,3', 'x=2', 'y=3'], '2,4': ['label=2,4', 'x=2', 'y=4', 'amenity=food'], '3,0': ['label=3,0', 'x=3', 'y=0'], '3,1': ['label=3,1', 'x=3', 'y=1'], '3,2': ['label=3,2', 'x=3', 'y=2'], '3,3': ['label=3,3', 'x=3', 'y=3', 'parking=garage'], '3,4': ['label=3,4', 'x=3', 'y=4'], '4,0': ['lab

In [14]:
print(city_map.tags["0,0"])
print(city_map.tags["4,1"])
print(city_map.tags["2,4"])

['label=0,0', 'x=0', 'y=0', 'landmark=start']
['label=4,1', 'x=4', 'y=1', 'amenity=food']
['label=2,4', 'x=2', 'y=4', 'amenity=food']


### Run UCS on ShortestPathProblem

Find the shortest path from $(0,0)$ to the nearest location tagged `amenity=food`. On our 5x5 grid, the two food locations are at $(4,1)$ and $(2,4)$.

In [15]:
problem = ShortestPathProblem(city_map, "0,0", "amenity=food")

In [16]:
from util import UniformCostSearch

problem = ShortestPathProblem(city_map, "0,0", "amenity=food")

ucs = UniformCostSearch(verbose=1)
ucs.solve(problem)

print("Actions:", ucs.actions)
print("Path cost:", ucs.path_cost)
print("States explored:", ucs.num_states_explored)

num_states_explored = 19
path_cost = 5.0
actions = ['0,1', '1,1', '2,1', '3,1', '4,1']
Actions: ['0,1', '1,1', '2,1', '3,1', '4,1']
Path cost: 5.0
States explored: 19


**Result:** UCS finds the path $\{(0,0) \to (0,1) \to (1,1) \to (2,1) \to (3,1) \to (4,1)\}$ with cost 5.0, exploring 19 states. It correctly reaches $(4,1)$ which is the nearest food location (Manhattan distance 5), rather than $(2,4)$ which is further (Manhattan distance 6).

### 2b. Stanford Map Scenarios

**Task:** Using the Stanford campus map, select two scenarios:

1. A start/end tag pair that produces a **useful insight** about campus travel.
2. A start/end tag pair where the minimum cost path **isn't desirable** -- explain whether this stems from incorrect modeling (missing tags, inaccurate distances, etc.).

*Note: This problem requires the Stanford map data (`stanford.json`) and the visualization tools. The synthetic grid tests above verify correctness of the implementation.*

### 2c. Negative Externalities of Widespread Deployment

**Task:** Discuss the negative externalities of deploying a shortest-path routing system at scale.

**Answer:**

When millions of users follow algorithmically optimal routes, the system's model of the world diverges from reality. Each driver treats road costs as fixed, but collectively they *change* the costs by creating congestion.

1. **Externality for users:** The routes suggested become self-defeating at scale -- if the algorithm routes thousands of drivers through the same "shortcut," that road becomes slower than the main route. Users experience worse travel times than the algorithm predicted because the model doesn't account for other users receiving the same recommendation.

2. **Externality for non-users:** Routing apps divert traffic through residential neighborhoods and side streets that were never designed for high throughput. Residents who don't use the app experience increased noise, pollution, and safety hazards from cut-through traffic they have no control over.

**Mitigation:** One approach is *congestion-aware routing* that models current traffic load and distributes drivers across multiple paths (a social-optimum allocation rather than individual-optimum). Another is geofencing sensitive areas to prevent routing through residential streets unless the driver's origin or destination is within that area.

---
## Problem 3: Finding Shortest Paths with Unordered Waypoints

Now we extend the shortest path problem: find the shortest path that visits a set of **waypoint tags** (in any order) before reaching the goal.

**Key insight from Lecture 5:** The state must encode not just the current location, but also *which waypoints have been visited*. We track this as a tuple in the state's `memory` field. A single location can satisfy multiple waypoint tags simultaneously.

### 3a. WaypointsShortestPathProblem Implementation

**Task:** Implement `WaypointsShortestPathProblem` with:
- **Input:** `city_map`, `start_location`, `waypoint_tags` (set of tags to visit), `end_tag`
- **Output:** shortest path that visits all waypoints in any order, ending at a location with `end_tag`
- **State:** `State(location, memory)` where `memory` is a tuple of visited waypoint tags
- **Requirement:** Optimize for a compact state space (correct asymptotic complexity required)

In [17]:
class WaypointsShortestPathProblem(SearchProblem):
    def __init__(self, city_map, start_location, end_tag, waypoint_tags):
        self.city_map = city_map
        self.start_location = start_location
        self.end_tag = end_tag
        self.waypoint_tags = waypoint_tags

    def start_state(self):
        done = []
        for tag in self.waypoint_tags:
            if tag in self.city_map.tags[self.start_location]:
                done.append(tag)
        return State(location=self.start_location, memory=tuple(sorted(done)))

    def successors(self, state):
        result = []
        for neighbor, distance in self.city_map.distances[state.location].items():
            done = set(state.memory)
            for tag in self.waypoint_tags:
                if tag in self.city_map.tags[neighbor]:
                    done.add(tag)
            new_memory = tuple(sorted(done))
            result.append(
                Step(
                    action=neighbor,
                    cost=distance,
                    state=State(location=neighbor, memory=new_memory)
                    )
                    )
        return result

    def is_end(self, state):
        return (
            self.end_tag in self.city_map.tags[state.location]
            and set(self.waypoint_tags).issubset(set(state.memory))
            )

**Implementation notes:**

- `start_state()` checks whether the start location already satisfies any waypoint tags (important edge case).
- `successors()` updates the `memory` tuple when a neighbor satisfies a waypoint tag. Using `tuple(sorted(...))` ensures a canonical representation so equivalent states hash identically.
- `is_end()` requires both: (1) current location has `end_tag`, and (2) all waypoint tags have been visited.

### Test on a Synthetic 6x6 Grid

Set up a grid where location $(2,2)$ satisfies *both* waypoint tags (`amenity=food` and `parking=garage`) simultaneously, with distractors at $(1,4)$ and $(4,0)$.

In [18]:
from map_util import create_grid_map_with_custom_tags, check_valid, get_total_cost
from util import UniformCostSearch, State

# 6x6 synthetic grid
tags = {(x, y): [] for x in range(6) for y in range(6)}

# Start and goal
tags[(0, 0)] = ["landmark=start"]
tags[(5, 2)] = ["landmark=goal"]

# Waypoints
# This one location satisfies BOTH waypoint tags at once
tags[(2, 2)] = ["amenity=food", "parking=garage"]

# Extra distractors so the map is not trivial
tags[(1, 4)] = ["amenity=food"]
tags[(4, 0)] = ["parking=garage"]

city_map = create_grid_map_with_custom_tags(6, 6, tags)

start_location = "0,0"
end_tag = "landmark=goal"
waypoint_tags = ["amenity=food", "parking=garage"]

problem = WaypointsShortestPathProblem(
    city_map=city_map,
    start_location=start_location,
    end_tag=end_tag,
    waypoint_tags=waypoint_tags,
)

ucs = UniformCostSearch(verbose=1)
ucs.solve(problem)

path = [start_location] + ucs.actions

print("Actions:", ucs.actions)
print("Path cost:", ucs.path_cost)
print("States explored:", ucs.num_states_explored)
print("Path:", path)
print("Valid path?", check_valid(path, city_map, start_location, end_tag, waypoint_tags))
print("Recomputed total cost:", get_total_cost(path, city_map))

num_states_explored = 73
path_cost = 7.0
actions = ['0,1', '0,2', '1,2', '2,2', '3,2', '4,2', '5,2']
Actions: ['0,1', '0,2', '1,2', '2,2', '3,2', '4,2', '5,2']
Path cost: 7.0
States explored: 73
Path: ['0,0', '0,1', '0,2', '1,2', '2,2', '3,2', '4,2', '5,2']
Valid path? True
Recomputed total cost: 7.0


**Result:** The path $\{(0,0) \to (0,1) \to (0,2) \to (1,2) \to (2,2) \to (3,2) \to (4,2) \to (5,2)\}$ has cost 7.0 and explores 73 states. The algorithm correctly discovers that visiting $(2,2)$ satisfies both waypoints at once, rather than making separate detours to $(1,4)$ and $(4,0)$.

### 3b. State Space Analysis

**Task:** With $n$ locations and $k$ waypoint tags, provide a mathematical expression for the maximum number of states UCS could visit.

**Answer:**

Each state is a `(location, memory)` pair where:
- `location` can be any of $n$ locations
- `memory` is a subset of the $k$ waypoint tags (represented as a sorted tuple)

The number of possible subsets of $k$ tags is $2^k$, so the maximum number of distinct states is:

$$n \cdot 2^k$$

This is why the problem specification emphasizes compact state representation -- if we tracked waypoints inefficiently (e.g., as an ordered visit sequence), the state space could explode to $n \cdot k!$ or worse. The subset representation is optimal because the *order* of waypoint visits doesn't matter, only *which* ones have been visited.

### 3c. Custom Waypoints Path

*This problem requires the Stanford map and visualization tools. The synthetic grid tests above verify correctness of the implementation.*

### 3d. Labor Practices Discussion

**Task:** In the context of ride-sharing, how can unordered waypoints improve driver experience while balancing company goals?

**Answer:**

**(a) Waypoints between drop-off and pick-up:** Include stops like gas stations, restrooms, affordable food options, and rest areas. Drivers often work long shifts with no built-in breaks, so routing through essential amenities between rides could meaningfully improve working conditions without significantly increasing pick-up ETAs.

**(b) Driver information needed:** To select appropriate waypoints, the system needs: time since last break, fuel/charge level, dietary preferences or restrictions, and driver-specified priorities (e.g., "I need a bathroom" vs "I need food"). Duration of current shift is also critical for safety-related rest stops.

**(c) Potential negative uses:** The same waypoint system could be used to route drivers through company-preferred partners (sponsored gas stations, restaurants) rather than the driver's best option. Collecting granular information about driver needs creates a power asymmetry -- the company could use break patterns, fatigue indicators, and location history to manipulate driver behavior (e.g., offering surge pricing right before a driver would normally stop, incentivizing them to skip breaks). The data could also be used to penalize drivers who take "too many" or "too long" breaks.

---
## Problem 4: Speeding up Search with A*

UCS explores states in order of past cost alone. A* improves on this by incorporating a **heuristic** $h(s)$ that estimates the future cost from state $s$ to the goal.

**From Lecture 6:** A* can be implemented as a *reduction* to UCS. Given original cost $\text{Cost}(s, a)$ and heuristic $h$, define modified costs:

$$\text{Cost}'(s, a) = \text{Cost}(s, a) + h(s') - h(s)$$

where $s'$ is the successor state. Running UCS with these modified costs is equivalent to A*. If $h$ is **consistent** (satisfies the triangle inequality), all modified costs remain non-negative and the reduction is valid.

### 4a. A* via Reduction to UCS

**Task:** Implement `a_star_reduction(problem, heuristic)` that returns a new `SearchProblem` with modified edge costs.

The key insight: rather than modifying UCS itself, we transform the *problem* so that ordinary UCS on the transformed problem behaves like A* on the original.

In [19]:
def a_star_reduction(problem, heuristic):
    class ReducedProblem(SearchProblem):
        def start_state(self):
            return problem.start_state()

        def is_end(self, state):
            return problem.is_end(state)

        def successors(self, state):
            result = []
            for step in problem.successors(state):
                new_cost = (
                    step.cost
                    + heuristic.evaluate(step.state)
                    - heuristic.evaluate(state)
                )
                result.append(
                    Step(
                        action=step.action,
                        cost=new_cost,
                        state=step.state
                    )
                )
            return result

    return ReducedProblem()

### Verify with Zero Heuristic

A zero heuristic ($h(s) = 0$ for all $s$) should produce exactly the same behavior as plain UCS, since the modified cost equals the original cost: $\text{Cost}'(s,a) = \text{Cost}(s,a) + 0 - 0$.

In [20]:
from util import Heuristic, UniformCostSearch

class ZeroHeuristic(Heuristic):
    def evaluate(self, state):
        return 0.0

zero_heuristic = ZeroHeuristic()

# Use one of your existing synthetic problems
problem = ShortestPathProblem(city_map, "0,0", "amenity=food")

# Ordinary UCS
ucs_original = UniformCostSearch(verbose=1)
ucs_original.solve(problem)

print("Original UCS actions:", ucs_original.actions)
print("Original UCS path cost:", ucs_original.path_cost)
print("Original UCS states explored:", ucs_original.num_states_explored)

# UCS on the A*-reduced problem
reduced_problem = a_star_reduction(problem, zero_heuristic)

ucs_reduced = UniformCostSearch(verbose=1)
ucs_reduced.solve(reduced_problem)

print("Reduced UCS actions:", ucs_reduced.actions)
print("Reduced UCS path cost:", ucs_reduced.path_cost)
print("Reduced UCS states explored:", ucs_reduced.num_states_explored)

num_states_explored = 13
path_cost = 4.0
actions = ['0,1', '0,2', '1,2', '2,2']
Original UCS actions: ['0,1', '0,2', '1,2', '2,2']
Original UCS path cost: 4.0
Original UCS states explored: 13
num_states_explored = 13
path_cost = 4.0
actions = ['0,1', '0,2', '1,2', '2,2']
Reduced UCS actions: ['0,1', '0,2', '1,2', '2,2']
Reduced UCS path cost: 4.0
Reduced UCS states explored: 13


**Result:** Identical actions, path cost, and states explored -- confirming the reduction is correctly implemented. The zero heuristic is a useful sanity check.

### 4b. StraightLineHeuristic

**Task:** Implement a heuristic that returns the straight-line (Euclidean) distance from the current location to the nearest goal location.

**Why it's admissible:** The straight-line distance is always $\leq$ the actual shortest path distance (you can't do better than a straight line), so it never overestimates. This makes it a valid A* heuristic.

**Efficiency note:** Precompute the set of goal locations in `__init__` so that `evaluate()` only needs to compute distances.

In [21]:
from util import Heuristic
from map_util import compute_distance

class StraightLineHeuristic(Heuristic):
    def __init__(self, city_map, end_tag):
        self.city_map = city_map
        self.end_tag = end_tag

        self.end_locations = []
        for location, tags in self.city_map.tags.items():
            if self.end_tag in tags:
                self.end_locations.append(location)

    def evaluate(self, state):
        current_geo = self.city_map.geo_locations[state.location]

        distances = []
        for end_location in self.end_locations:
            end_geo = self.city_map.geo_locations[end_location]
            distances.append(compute_distance(current_geo, end_geo))

        return min(distances)
        

### Test StraightLineHeuristic

Verify that the heuristic returns sensible values: positive for locations far from the goal, zero at the goal itself.

In [26]:
from map_util import create_grid_map_with_custom_tags

tags = {(x, y): [] for x in range(6) for y in range(6)}
tags[(0, 0)] = ["landmark=start"]
tags[(5, 2)] = ["landmark=goal"]
tags[(2, 2)] = ["amenity=food", "parking=garage"]
tags[(1, 4)] = ["amenity=food"]
tags[(4, 0)] = ["parking=garage"]

city_map = create_grid_map_with_custom_tags(6, 6, tags)

heuristic = StraightLineHeuristic(city_map, "landmark=goal")
print(heuristic.evaluate(State(location="0,0")))
print(heuristic.evaluate(State(location="5,2")))

5.9880300569817
0.0


**Result:** Heuristic returns ~5.99 for $(0,0)$ (far from goal at $(5,2)$) and 0.0 for $(5,2)$ (at the goal). Both are correct.

### Run A* with Straight-Line Heuristic

Compare A* (via reduction) against plain UCS on the same shortest path problem.

In [28]:
straight_line_heuristic = StraightLineHeuristic(city_map, "amenity=food")
reduced_problem = a_star_reduction(problem, straight_line_heuristic)

ucs_astar = UniformCostSearch(verbose=1)
ucs_astar.solve(reduced_problem)

print("A* via reduction actions:", ucs_astar.actions)
print("A* via reduction reduced-cost:", ucs_astar.path_cost)
print("A* via reduction states explored:", ucs_astar.num_states_explored)

from map_util import get_total_cost
path = [problem.start_state().location] + ucs_astar.actions
print("Returned path:", path)
print("Original path cost:", get_total_cost(path, city_map))

num_states_explored = 6
path_cost = 0.8549325334437041
actions = ['1,0', '1,1', '2,1', '2,2']
A* via reduction actions: ['1,0', '1,1', '2,1', '2,2']
A* via reduction reduced-cost: 0.8549325334437041
A* via reduction states explored: 6
Returned path: ['0,0', '1,0', '1,1', '2,1', '2,2']
Original path cost: 4.0


**Result:** A* explores only **6 states** compared to UCS's 13 -- a 54% reduction. The heuristic guides search toward the goal, avoiding exploration of states in the wrong direction. Both find the same optimal path with cost 4.0.

Note the reduced cost (0.85) differs from the original cost (4.0) because the A* reduction transforms edge weights. The *actions* and *original path cost* are what matter.

### 4c. NoWaypointsHeuristic

**Task:** Implement a heuristic that returns the shortest path cost to the goal **ignoring all waypoints**. This is a relaxation -- removing constraints can only decrease cost, so the heuristic is admissible.

**Efficiency note:** Precompute the shortest path cost from *every* location to the goal (by solving a `ShortestPathProblem` from each location). This makes `evaluate()` a simple dictionary lookup at query time.

In [29]:
from util import Heuristic, UniformCostSearch

class NoWaypointsHeuristic(Heuristic):
    def __init__(self, city_map, end_tag):
        self.city_map = city_map
        self.end_tag = end_tag
        self.cost_to_goal = {}

        for location in self.city_map.geo_locations.keys():
            problem = ShortestPathProblem(
                city_map=self.city_map,
                start_location=location,
                end_tag=self.end_tag
            )
            ucs = UniformCostSearch(verbose=0)
            ucs.solve(problem)
            self.cost_to_goal[location] = ucs.path_cost

    def evaluate(self, state):
        return self.cost_to_goal[state.location]

### Test NoWaypointsHeuristic

Verify the precomputed costs: $(0,0)$ should be far from the goal, $(5,2)$ should have cost 0 (it *is* the goal), and $(2,2)$ should be somewhere in between.

In [30]:
no_wp_heuristic = NoWaypointsHeuristic(city_map, "landmark=goal")

print(no_wp_heuristic.evaluate(State(location="0,0")))
print(no_wp_heuristic.evaluate(State(location="5,2")))
print(no_wp_heuristic.evaluate(State(location="2,2")))

7.0
0.0
3.0


**Result:** Costs of 7.0, 0.0, and 3.0 for the three locations -- all match the true shortest path distances (ignoring waypoints) on the 6x6 grid. The heuristic is correctly precomputed.

### Compare All Three Approaches on the Waypoints Problem

Run the full waypoints problem with: (1) plain UCS, (2) A* with straight-line heuristic, (3) A* with no-waypoints heuristic.

In [31]:
waypoint_problem = WaypointsShortestPathProblem(
    city_map=city_map,
    start_location="0,0",
    end_tag="landmark=goal",
    waypoint_tags=["amenity=food", "parking=garage"]
)

# 1) Plain UCS
ucs_plain = UniformCostSearch(verbose=1)
ucs_plain.solve(waypoint_problem)

print("Plain UCS actions:", ucs_plain.actions)
print("Plain UCS reduced/original cost:", ucs_plain.path_cost)
print("Plain UCS explored:", ucs_plain.num_states_explored)

# 2) A* with straight-line heuristic
sl_heuristic = StraightLineHeuristic(city_map, "landmark=goal")
sl_reduced = a_star_reduction(waypoint_problem, sl_heuristic)

ucs_sl = UniformCostSearch(verbose=1)
ucs_sl.solve(sl_reduced)

print("A* straight-line actions:", ucs_sl.actions)
print("A* straight-line reduced cost:", ucs_sl.path_cost)
print("A* straight-line explored:", ucs_sl.num_states_explored)

# 3) A* with no-waypoints heuristic
nw_heuristic = NoWaypointsHeuristic(city_map, "landmark=goal")
nw_reduced = a_star_reduction(waypoint_problem, nw_heuristic)

ucs_nw = UniformCostSearch(verbose=1)
ucs_nw.solve(nw_reduced)

print("A* no-waypoints actions:", ucs_nw.actions)
print("A* no-waypoints reduced cost:", ucs_nw.path_cost)
print("A* no-waypoints explored:", ucs_nw.num_states_explored)

# Recompute original path costs for the two A* runs
from map_util import get_total_cost

path_sl = ["0,0"] + ucs_sl.actions
path_nw = ["0,0"] + ucs_nw.actions

print("A* straight-line original cost:", get_total_cost(path_sl, city_map))
print("A* no-waypoints original cost:", get_total_cost(path_nw, city_map))

num_states_explored = 73
path_cost = 7.0
actions = ['0,1', '0,2', '1,2', '2,2', '3,2', '4,2', '5,2']
Plain UCS actions: ['0,1', '0,2', '1,2', '2,2', '3,2', '4,2', '5,2']
Plain UCS reduced/original cost: 7.0
Plain UCS explored: 73
num_states_explored = 23
path_cost = 1.0119699430182998
actions = ['1,0', '2,0', '2,1', '2,2', '3,2', '4,2', '5,2']
A* straight-line actions: ['1,0', '2,0', '2,1', '2,2', '3,2', '4,2', '5,2']
A* straight-line reduced cost: 1.0119699430182998
A* straight-line explored: 23
num_states_explored = 24
path_cost = 0.0
actions = ['0,1', '0,2', '1,2', '2,2', '3,2', '4,2', '5,2']
A* no-waypoints actions: ['0,1', '0,2', '1,2', '2,2', '3,2', '4,2', '5,2']
A* no-waypoints reduced cost: 0.0
A* no-waypoints explored: 24
A* straight-line original cost: 7.0
A* no-waypoints original cost: 7.0


**Analysis:**

| Method | States Explored | Original Path Cost | Optimal? |
|--------|:-:|:-:|:-:|
| Plain UCS | 73 | 7.0 | Yes |
| A* + Straight-Line | 23 | 7.0 | Yes |
| A* + No-Waypoints | 24 | 7.0 | Yes |

All three find the optimal path with cost 7.0, but A* dramatically reduces the search space:

- **Straight-line heuristic** cuts exploration by **68%** (73 → 23 states). It provides directional guidance toward the goal.
- **No-waypoints heuristic** cuts exploration by **67%** (73 → 24 states). It uses *exact* shortest-path distances (ignoring waypoints) as the estimate, which is a tighter bound.

On this small grid the two A* heuristics perform similarly, but on larger maps the no-waypoints heuristic is generally stronger because it accounts for actual graph structure rather than just geometric distance. The trade-off is precomputation cost: the no-waypoints heuristic solves $n$ shortest-path problems upfront.

---
## Problem 5: AI Tutor

**5a.** Dynamic Programming tutoring session (~15-20 minutes with an AI chatbot).

**5b.** Beam Search tutoring session (~15-20 minutes with an AI chatbot).

*These are interactive sessions conducted externally. Deliverable: links to chat transcripts.*

---
## Key Takeaways

### From Lecture 5 (applied in Problems 1, 2 & 3)
- **Search problems** are defined by states, actions, costs, and goal tests -- the abstraction is powerful enough to model route planning, waypoint visiting, and more
- **UCS** guarantees optimal paths when costs are non-negative, and terminates even on infinite state spaces (as long as costs are bounded away from zero)
- **State design** is critical: for waypoints, encoding visited tags as a sorted tuple gives a state space of $n \cdot 2^k$ rather than the exponentially worse $n \cdot k!$
- Real-world deployment of routing algorithms creates **externalities** (congestion, residential cut-through traffic) that the model doesn't capture

### From Lecture 6 (applied in Problem 4)
- **A* search** uses a heuristic $h(s)$ to guide exploration toward the goal, often dramatically reducing the number of states explored
- A* can be implemented as a **reduction to UCS** by modifying edge costs: $\text{Cost}'(s,a) = \text{Cost}(s,a) + h(s') - h(s)$
- **Admissibility** ($h(s) \leq$ true cost) guarantees optimality; **consistency** ($h(s) \leq \text{Cost}(s,a) + h(s')$) guarantees non-negative modified costs
- **Relaxation** is a powerful technique for constructing heuristics: remove constraints (like waypoints) to get a simpler problem whose optimal cost is an admissible lower bound
- Better heuristics explore fewer states but may cost more to compute -- there is always a **precomputation vs. search trade-off**